# Preparation Handbook

**Common Approaches to Data Preparation**

*Session 5: Data Preparation*

## Introduction

This notebook is a **handbook**: a catalogue of the problems that show up when preparing real data, and the standard ways of handling each one. You consult it while building your own preparation script — you don't fill it in and hand it back.

That makes it deliberately different from Session 4's notebook:

| | Session 4 — diagnostic **template** | Session 5 — preparation **handbook** (this file) |
|---|---|---|
| How you use it | duplicate it, fill it in | look up the entry you need |
| Why | every dataset gets the same checks | you only prepare what your diagnosis found |
| What you hand in | the completed notebook | your preparation **script** — not this file |

**Scope**: turning diagnosed problems into justified fixes. Not diagnosis (Session 4 did that), and not exploration (Sessions 6–8 do that). You *will* come back and change this work once EDA shows you something new — that loop is expected, not a sign you did it wrong.

**Two artifacts, two jobs**

| | Notebook (this file) | Script (your Session 5 deliverable) |
|---|---|---|
| Purpose | experiment, inspect, undo, retry | reproduce the final result |
| Audience | you, while you decide | anyone who needs the clean data |
| Contains | dead ends, checks, justifications | only the decisions that survived |

**How to use this handbook**

- 🔧 **Recipe** cells are working code on a real column — copy one, swap in your own column, run it
- 💡 **Example** cells show how it played out on this course's birth-records dataset — read them, don't just copy them
- 🧱 **Script** marks logic that survives into your preparation script. Each of these cells is self-contained: copy them in order and you have a working script
- 🔎 **Check** cells confirm the action did what you expected. They belong in the notebook and **never** in the script — a script has nobody to show output to
- Sections **3–8 are lookup entries**: go to the ones your diagnosis points at, skip the rest (0.4 tells you which)
- Sections **0–2 and 9 are not optional** — they are the spine every preparation follows, whatever your dataset turned out to need

**Contents**

0. From Diagnosis to Preparation Plan
1. Setup & Ground Rules
2. Structural Fixes
3. Unmasking Hidden Missing Values
4. Nominal Variables
5. Ordinal Variables
6. Numeric Variables
7. Temporal Variables
8. Cross-Field Consistency
9. Export & Handoff to the Script

# 0. From Diagnosis to Preparation Plan

## 0.1 What You Bring From Session 4

Two things carry over from the diagnostic notebook:

1. Your **business/analytical question**, and
2. Your **diagnosis report** — the markdown file you submitted at the end of Session 4: one row per column that needs attention, with evidence, a severity and an action.

Restate both here so every decision below can be checked against them.

> *Your question, in one line.*

> *Your diagnosis in one line: how many columns need attention, and which one worries you most?*

...

## 0.2 A Diagnosis Is Not a Plan

A diagnosis says *what is wrong*. A plan says *what you will do about it, and why*. The gap between the two is where most of the thinking in this session happens.

For every finding you have three defensible answers, and "fix it" is only one of them:

| Decision | When it applies | What it costs |
|---|---|---|
| **Repair** | you know what the correct value should be, or you have a defensible substitute | it writes your assumption into the data |
| **Flag** | something is clearly wrong, but you can't tell what "right" would be | keeps the row usable, pushes the decision downstream |
| **Leave** | the value is odd but real, or simply irrelevant to your question | nothing — as long as it's a deliberate choice, not an oversight |

Deleting rows is not a fourth option. It's the most expensive form of repair, because it's the only one nobody downstream can undo.

The default is **not** "clean everything." The default is: *does this issue threaten my ability to answer my question?* A perfectly cleaned column you never use is wasted work; a quietly imputed column you build a conclusion on is worse than wasted.

...

## 0.3 Ground Rules

1. **Never modify the raw data.** `data/raw/` is immutable. Everything happens on a copy.
2. **One working copy, transformed in a known order.** Not five half-cleaned DataFrames you lose track of.
3. **Every change gets a written reason.** If you can't say why, don't do it yet.
4. **Re-run top to bottom before you believe anything.** Notebooks let you run cells out of order; your script won't.
5. **Order matters.** Preparation steps are not independent — several of them change what the next check sees.

The order used in this notebook, and why:

```
unmask hidden missing values   →  because missingness counts are wrong until you do
structural fixes               →  drop what carries no information before working on it
text hygiene                   →  make labels comparable before comparing them
category consolidation         →  merge by meaning, once the text is comparable
plausibility repair            →  turn impossible values into missing ones...
outlier decisions              →  ...so outlier logic only sees values that could be real
imputation                     →  fill last, once you know what is genuinely missing
derived features               →  build on prepared columns, not raw ones
export                         →  hand off to EDA and to your script
```

Imputing before repairing implausible values, for instance, bakes a `100`-year-old parent into the median you impute with. The order isn't ceremonial.

...

## 0.4 Finding Your Way In

Take your diagnosis report one finding at a time and use this table to find the entry that applies. Rows your diagnosis never raised are rows you skip — that is what makes this a handbook rather than a checklist.

| What your diagnosis said | Where to go |
|---|---|
| "a single value in every row" | 2.2 Columns With No Information |
| "duplicate records" | 2.3 Duplicates |
| "`SIN INFORMACION` / `N/A` / `9999` used as a category" | 3.1 Missingness in Disguise |
| "column X is Y% missing" | 3.2 Why Is It Missing? → then 4.5 (text) or 6.4 (numeric) |
| "the diagnosis itself looks wrong now" | 3.3 When the Diagnosis Was Wrong |
| "inconsistent casing, whitespace or accents" | 4.1 Text Hygiene |
| "two labels mean the same thing" / "too many categories" | 4.3 Consolidating Categories by Meaning |
| "a category with only a handful of rows" | 4.4 Rare Categories |
| "missing values in a text column" | 4.5 Missing Values in Nominal Columns |
| "this column's categories have a meaningful order" | 5.1 Declaring the Order |
| "a numeric column that is really a label or a scale" | 5.3 Ordinal Variables That Look Numeric |
| "values outside a plausible range" | 6.1 Plausibility Repair |
| "extreme values / outliers" | 6.2 Outliers Are Not Errors |
| "missing numbers" | 6.4 Missing Numeric Values |
| "dates stored as text" | 7.1 Parsing |
| "column A contradicts column B" | 8. Cross-Field Consistency |

**Three entries are not diagnosis-driven**, and you should visit them whichever findings you brought:

- **6.2 Outliers** — an outlier is a statement about a *distribution*, and the distribution only becomes readable once the disguised missing values are unmasked and the categories are merged. Session 4 could flag values that are *impossible*; it could not flag values that are merely extreme. Expect to meet these here for the first time.
- **6.4 Missing Numeric Values** — Section 3 changes the missingness numbers, sometimes drastically, so the picture you diagnosed is not the picture you are now preparing.
- **8. Cross-Field Consistency** — rules between columns are easy to overlook when you are inspecting one column at a time.

Anything you find in those entries is a genuinely *new* finding. It does not go back into your diagnosis report — it goes in **section 2b of your preparation record**, along with an honest note about whether Session 4 could have caught it.

Whatever you use or skip, **9. Export & Handoff** always runs.

# 1. Setup & Ground Rules

## 1.1 Load the Raw Data

In [ ]:
# 🧱 Script
import numpy as np
import pandas as pd

In [ ]:
# 🔎 Check — a display setting for this notebook; a script has nothing to display
pd.set_option("display.max_columns", 40)

In [ ]:
# 🧱 Script — point this at your own dataset
RAW_PATH = "../data/raw/nacimientos_ocurridos_hospital_general_medellin.csv"

df_raw = pd.read_csv(RAW_PATH, sep=",", header="infer")

In [ ]:
# 🔎 Check
df_raw.shape

> ⚠️ `df_raw` is read-only from here on. Every transformation happens on the working copy below.

## 1.2 The Working Copy

In [ ]:
# 🧱 Script
df_prep = df_raw.copy()

In [ ]:
# 🔎 Check
df_prep.head(3)

One copy, not many. When an experiment goes wrong, re-run this cell and start the section again — that is exactly the kind of cheap undo a notebook gives you and a script does not.

...

## 1.3 Your Preparation Record

Everything you decide from here on gets written into one document: your **preparation record**. Together with the script, it is what you submit for this session — the notebook itself is not handed in.

Copy `templates/4-preparation_record_template.md` now and rename it for your dataset. Its input is the diagnosis report you submitted at the end of Session 4.

**You do not fill it in one sitting.** Each of its five sections is written at a different moment, and writing them at the right moment is most of what makes the record honest — a plan written afterwards is not a plan, and a log written from memory is a story. Here is the whole document and when each part gets filled:

| Section of the record | What goes in it | When you fill it | Where in this handbook |
|---|---|---|---|
| **1. Plan** | one row per finding in your diagnosis report, each resolved as repair / flag / leave | **before** you write any cleaning code | 1.3 — now |
| **2a. Did not survive** | findings that looked like defects and turned out legitimate | the moment one turns up | 3.3 |
| **2b. Emerged during preparation** | problems your diagnosis could not have seen yet | the moment one turns up | 0.4, 6.2, 6.4, 8 |
| **3. Log** | what you actually did and why — including "nothing, deliberately" | as you work, never from memory | every 🧱 step; collected in 9.2 |
| **4. Plan vs. log** | where the log departs from the plan, and what changed your mind | at the end | 9.2 |
| **5. Result** | rows and columns in and out, missingness left on purpose, what you checked before exporting | at the end | 9.1, 9.2 |

**One rule holds throughout: you never edit the diagnosis report.** It was submitted; it stands. Everything that revises it goes in section 2 of this record. The gap between what you suspected in Session 4 and what you concluded in Session 5 is evidence of thinking, not an error to tidy away.

### Section 1: the plan

Turn each finding in your diagnosis report into a decision *before* you write cleaning code. Writing it down first is what stops "cleaning" from becoming an unexamined series of habits.

One plan row per finding, in the same order as the report, so anyone can lay the two documents side by side and see that nothing was quietly dropped. Your report holds between five and ten findings, so the plan has exactly that many rows — the triage already happened in Session 4, and nothing gets dropped on the way here.

**Your plan will not survive contact with the data, and it isn't supposed to.** It covers the findings you arrived with; preparation will hand you more, because some problems are only visible once others are fixed. Those go in section 2b as they surface — not back into this plan, which stays a record of what you intended *before* you started. **Section 2b is not bounded by the 5–10 range**: that range governed the diagnosis, where the job was to prioritise, and dropping a real finding here just to stay under a number would be exactly the wrong instinct. If 2b ends up with more than about five rows, though, that says something about how carefully Session 4 went — worth admitting in section 4 of the record.

💡 **Example** — section 1 of the preparation record, for the birth-records dataset:

| # | Column | Issue (from the diagnosis) | Decision | Method | Why |
|---|---|---|---|---|---|
| 1 | `pais_residencia`, `profesion_certificador` | a single value in every row | repair | drop the columns, record what they said as scope facts | no information to analyse; the fact itself belongs in the scope note |
| 2 | `localidad`, `nivel_educativo_padre`, … | `SIN INFORMACION` used as a category | repair | recode to `NaN` | missingness in disguise; leaving it inflates every category count |
| 3 | `sexo` | `INDETERMINADO` in 1 row | **leave** | keep it as a category | a real clinical category, not a missing-value code — see 3.3 |
| 4 | `pertenencia_etnica` | two labels for one group, punctuation differs | repair | consolidate by meaning | otherwise the same group is split in every grouped analysis |
| 5 | `estado_conyugal_madre` | 13 labels for ~6 real states | repair | explicit label mapping | past/present wording is a form artifact, not a difference in marital status |
| 6 | `edad_padre` | implausible ages (94, 100); 4.5% missing | repair + **flag** | implausible → `NaN`; add a missing indicator | the true age is unrecoverable, and the missingness may itself be informative |
| 7 | `peso_gramos` | 48 values under 1,000 g | **leave** | none | corroborated by gestation weeks — real preterm births, and the population the question is about |
| 8 | `ultimo_ano_aprobado_madre` | 1.8% missing | repair | structural fill by education level | missing exactly where "no education" makes the value zero — not random |
| 9 | `numero_hijos_nacidos_vivos` vs `numero_embarazos` | 116 rows with more live births than pregnancies | **flag** | boolean flag column | impossible, but nothing says which of the two fields is wrong |

Notice that three of the nine rows are **not** repairs. A plan where everything says "repair" usually means the alternatives were never considered.

### Keeping the log as you go

Section 3 is the one people get wrong, because it is the only one that cannot be written at the end. Every time you change the data, note three things immediately: **the column, what you did, and why**. That is the whole log; the record is just where those notes get collected in 9.2.

If you find yourself reconstructing what you did an hour ago, the log has already failed — you will remember the actions and forget the reasons, and the reasons are the part that gets graded.

...

# 2. Structural Fixes

Structure before content: these steps change *which columns and rows exist*, so everything after them works on a smaller, cleaner surface.

## 2.1 Column Names

Session 4 flagged any stray whitespace or inconsistent casing in the column names. Fix it once, here, so every later reference is predictable.

In [ ]:
# 🧱 Script — normalize column names
df_prep.columns = (
    df_prep.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

In [ ]:
# 🔎 Check — did any name actually change?
df_prep.columns.tolist()

> Observations:

* Did any column name actually change?
* If names changed, does any code you already wrote still refer to the old ones?

...

## 2.2 Columns With No Information

A column with a single distinct value cannot explain anything — it has no variation to relate to anything else. Drop it, but **write down what it said**: "every record is from Colombia" is a fact about the *scope* of your dataset, and it belongs in your scope note even after the column is gone.

In [ ]:
# 🧱 Script — the columns are computed, not hard-coded, so this keeps working
# when the extract changes.
constant_cols = df_prep.columns[df_prep.nunique(dropna=False) <= 1].tolist()

df_prep = df_prep.drop(columns=constant_cols)

In [ ]:
# 🔎 Check
print("dropped:", constant_cols)
df_prep.shape

> 💡 Example (birth data): `pais_residencia` is always `COLOMBIA` and `profesion_certificador` is always `MEDICO`. Dropping them loses nothing — *provided* the scope note now says: this extract covers Colombian residents only, and every birth was certified by a physician.

...

## 2.3 Duplicates

Session 4 counted duplicates; here you act on them. Two different questions, two different checks:

- **Full-row duplicates** — the same record loaded twice.
- **Key duplicates** — two records that claim to describe the same real-world thing, but disagree in some column. These are the dangerous ones, because deduplicating means *choosing which version is true*.

In [ ]:
# 🔧 Recipe — swap in the columns that identify one record in your data.
# 💡 Birth data has no ID column, so this uses a natural key: attributes whose
# combination should not repeat by chance.
key_cols = ["fecha_nacimiento", "sexo", "peso_gramos", "talla_centimetros", "edad_madre"]

print("full-row duplicates:", df_prep.duplicated().sum())
print("key duplicates:", df_prep.duplicated(subset=key_cols).sum())

**Look before you delete.** If the counts above are not zero, put the copies side by side first — `keep=False` marks *every* member of each duplicate group, not just the repeats, which is what lets you see how the versions differ.

In [ ]:
# 🔧 Recipe — inspect the duplicate groups before deciding anything.
duplicate_groups = df_prep[df_prep.duplicated(subset=key_cols, keep=False)]

duplicate_groups.sort_values(key_cols).head(10)

Then act. Exact repeats are the easy case: if two rows agree in every column, keeping one loses nothing.

In [ ]:
# 🧱 Script — drop exact repeats.
df_prep = df_prep.drop_duplicates()

In [ ]:
# 🔎 Check — rows lost so far, measured against the raw file
print(f"{len(df_raw)} raw rows -> {len(df_prep)} now")

> 💡 Example (birth data): this removes nothing — there were no duplicates to begin with. Keep the line in your script anyway: it costs one pass over the data and it protects you when the extract is refreshed and someone appends the same file twice. **A no-op today is not a wasted line; it is a guarantee that holds tomorrow.**

Key duplicates are the harder case, and there is no default answer — deduplicating means deciding *which version is true*:

In [ ]:
# 🔧 Recipe — keeping one row per key. Choose the rule deliberately:
#   keep="first"  → the earliest row in file order
#   keep="last"   → the latest, if the file is ordered by update time
# Or sort first, so that "first" means what you actually want:
#   df_prep = df_prep.sort_values("fecha_actualizacion")

# df_prep = df_prep.drop_duplicates(subset=key_cols, keep="last")

> ⚠️ Not every repeated row is an error. Repeated transactions, sensor readings at the same timestamp, or a survey a household answered twice can all be legitimate. Blanket deduplication destroys those. Include the cell above in your script only once your check has shown the repeats are defects.

> Observations:

* Are there duplicates at all? (This dataset has none of either kind — a clean result is still a result, and it goes in your log as "checked, none found.")
* If your dataset has key duplicates, which record would you keep, and on what grounds — most recent, most complete, first seen?
* If you keep them, say why they are legitimate repetitions rather than errors.

...

# 3. Unmasking Hidden Missing Values

**When you need this**: always — run it even if your diagnosis didn't flag it, because it changes the numbers every later entry depends on.

This section comes before everything type-specific for one reason: **your missingness numbers are wrong until it runs.**

## 3.1 Missingness in Disguise

`isna()` only finds the missing values the file bothered to mark as missing. Real-world extracts — government open data especially — mark them a second way: as a *category* that means "we don't know."

`SIN INFORMACION`, `NO REPORTA`, `N/A`, `DESCONOCIDO`, `9999`, `-1`, empty strings. To pandas these are ordinary values. To you they are missing data, and every category count, every percentage, and every "most common value" is wrong while they are still counted as real.

In [ ]:
# 🧱 Script — adapt this list to the codes your own dataset uses.
MISSING_CODES = [
    "SIN INFORMACION",
    "NO REPORTA",
    "DESCONOCIDO",
    "NO APLICA",
    "N/A",
    "NA",
    "",
]

In [ ]:
# 🔎 Check — which columns actually contain them, and how often?
text_cols = df_prep.select_dtypes(include=["str"]).columns

found = df_prep[text_cols].isin(MISSING_CODES).sum()
found[found > 0].sort_values(ascending=False)

Before recoding them, compare the two pictures — that contrast is the whole point of this section.

In [ ]:
# 💡 Example (birth data): missingness as reported by isna()
(100 * df_prep.isna().mean()).sort_values(ascending=False).head(8).round(2)

In [ ]:
# 🧱 Script — recode disguised missing values to NaN
df_prep = df_prep.replace(MISSING_CODES, np.nan)

In [ ]:
# 💡 Example (birth data): the same columns, after unmasking
(100 * df_prep.isna().mean()).sort_values(ascending=False).head(8).round(2)

> 💡 Example (birth data): `localidad` goes from 3.6% missing to about 29.6%, and `nivel_educativo_padre` from 0% to 19.4% — it never had a single `NaN`, it had 1,220 rows saying `SIN INFORMACION`. Any conclusion about fathers' education drawn before this cell ran would have been drawn on a fifth of the data being fictional.

> Observations:

* Which columns changed the most between the two views?
* Does any column now cross a threshold where you'd reconsider using it at all?

...

## 3.2 Why Is It Missing?

Filling a gap before you know why it exists is how you invent data. Three causes, three different correct responses:

| Cause | Looks like | Correct response |
|---|---|---|
| **Structurally not applicable** | missing exactly where another column makes it meaningless | fill with the value implied by the structure (often `0`), or leave and exclude |
| **Missing at random** | scattered, small, unrelated to other columns | impute (median / mode), or drop rows if very few |
| **Informative** | missingness itself tracks something real | keep the `NaN`, add an indicator column |

Session 4's missingness-pattern check (`isnull().groupby(...)`) is what tells these apart. Re-run it now, on the unmasked data — the answers can change once the disguised values are counted.

In [ ]:
# 🔧 Recipe — does missingness in one column cluster inside another column's
# categories? Swap in a column with gaps and one you suspect explains them.
# 💡 Birth data: is the mother's last approved school year missing at random?
target_col = "ultimo_ano_aprobado_madre"
group_col = "nivel_educativo_madre"

(100 * df_prep[target_col].isna().groupby(df_prep[group_col]).mean()).sort_values(ascending=False).round(2)

> 💡 Example (birth data): the result is not "scattered." It is **100% missing** for mothers whose education level is `NINGUNO`, 100% for those whose level is now `NaN`, and essentially 0% everywhere else. That is not random missingness — for `NINGUNO` the true value is *zero years approved*, and the form simply left it blank. A global median would have filled those rows with 8 years of schooling that never happened.

> Observations:

* For each column with missing values: structural, random, or informative?
* Which of your columns would a global median or mode quietly falsify?

...

## 3.3 When the Diagnosis Was Wrong

The diagnostic notebook proposes; preparation decides. A finding that looked like a defect in Session 4 sometimes turns out to be legitimate once you look at it with the intent to act.

> 💡 Example (birth data): Session 4 flagged `sexo = "INDETERMINADO"` (1 row) as *"likely a missing-value code."* It isn't — indeterminate sex at birth is a real clinical category, recorded deliberately. So the decision here is **leave**, not recode. The Session 4 diagnosis stands as a record of what was suspected; this notebook records what was concluded, and why.

> Your turn: is there a finding in your diagnosis report that doesn't survive a second look? It goes in **section 2a of your preparation record** — never back into the report itself.

...

# 4. Nominal Variables

**When you need this**: your diagnosis flagged inconsistent labels, near-duplicate categories, a suspiciously high category count, or missing values in a text column.

Unordered categories: `sexo`, `tipo_parto`, `municipio_residencia`. Two failure modes matter here — labels that *look* different but mean the same thing, and labels that mean different things but get lumped together.

## 4.1 Text Hygiene

Case and stray whitespace create categories that only a computer can tell apart: `"MEDICO"`, `"medico"`, and `"MEDICO "` are three different values to pandas and one value to you. Normalize first, so that when two labels remain distinct you know it is a difference in *meaning*, not in typing.

In [ ]:
# 🧱 Script — apply to every text column
text_cols = df_prep.select_dtypes(include=["str"]).columns

for col in text_cols:
    df_prep[col] = df_prep[col].str.strip().str.upper()

In [ ]:
# 🔎 Check — did any column lose distinct values?
df_prep[text_cols].nunique().sort_values(ascending=False).head(10)

Two lines, and they are the ones you will reach for in almost every dataset you ever clean.

**Accents are the harder cousin of this problem** — `"INDÍGENA"` and `"INDIGENA"` are also one category typed two ways, but `.str.upper()` won't merge them. This dataset arrives unaccented, so it doesn't come up here; if yours does, the Appendix at the end of this notebook has a ready-made function for it.

> Observations:

* Did the number of distinct categories drop in any column? By how much?
* If nothing changed, was this dataset already clean, or did you normalize a column that didn't need it?

...

## 4.2 Hygiene Is Not Meaning

Normalizing text fixes *typing* differences. It does not fix *wording* differences — and those are much more common.

In [ ]:
# 💡 Example (birth data): two labels survive normalization
df_prep["pertenencia_etnica"].value_counts()

> 💡 Example (birth data): `NEGRO(A), MULATO(A), AFRO COLOMBIANO(A) O AFRO DESCENDIENTE` (81 rows) and `NEGRO(A) MULATO(A) AFRO COLOMBIANO(A) O AFRO DESCENDIENTE` (15 rows) differ by two commas. Same group, two versions of the form. Upper-casing and trimming cannot merge them — the difference is punctuation, not typing — so this needs a decision about *meaning*, which is the next subsection.

The lesson generalizes: after hygiene, any remaining duplicate-looking category is telling you that two humans described the same thing differently. No amount of string cleaning will decide whether they meant the same thing. You have to.

...

## 4.3 Consolidating Categories by Meaning

When several labels describe one real-world state, merge them with an explicit dictionary: `{old label: new label}`.

Writing every label out by hand is not laziness avoided — it *is* the method. You have to look at each label to decide where it goes, and anyone reviewing your work can see the entire merge at a glance.

In [ ]:
# 🧱 Script — the recipe form: swap in your own column and its mapping.
# 💡 Birth data: this closes the case from 4.2 — two labels for one group,
# differing only in punctuation.
variable = "pertenencia_etnica"

category_map = {
    "NEGRO(A) MULATO(A) AFRO COLOMBIANO(A) O AFRO DESCENDIENTE":
        "NEGRO(A), MULATO(A), AFRO COLOMBIANO(A) O AFRO DESCENDIENTE",
}

df_prep[variable] = df_prep[variable].replace(category_map)

In [ ]:
# 🔎 Check — five labels become four, and the merged group gains 15 rows
df_prep[variable].value_counts()

In [ ]:
# 💡 Example (birth data): the mother's marital status
df_prep["estado_conyugal_madre"].value_counts(dropna=False)

Read that list before writing any code. The twelve remaining labels (the `SIN INFORMACION` ones became `NaN` back in Section 3) are really **six states, each written twice** — once in the present tense and once in the past, because the form was reworded at some point. That is a difference in paperwork, not in marital status.

**The tempting shortcut, and why it's wrong.** All those labels contain the word `CASAD`, so it looks like one substring rule could do the job:

In [ ]:
# 💡 Example (birth data): how many rows contain "CASAD"?
df_prep["estado_conyugal_madre"].str.contains("CASAD", na=False).sum()

In [ ]:
# 💡 Example (birth data): how many mothers were actually married?
df_prep["estado_conyugal_madre"].isin(["ESTA CASADA", "ESTABA CASADO(A)"]).sum()

4,837 versus 565. Every *union libre* label contains the phrase `NO ESTABA CASADO(A)` — "was **not** married" — so the shortcut would have reclassified a third of the dataset as married, and produced a perfectly plausible-looking table while doing it.

This is the single most common way category cleaning goes wrong: a rule that matches on part of a label, written without reading all the labels first.

In [ ]:
# 🧱 Script
category_map = {
    "NO ESTABA CASADO(A) Y LLEVABA DOS ANOS O MAS VIVIENDO CON SU PAREJA": "UNION LIBRE >= 2 ANOS",
    "NO ESTA CASADA Y LLEVA DOS ANOS O MAS VIVIENDO CON SU PAREJA": "UNION LIBRE >= 2 ANOS",
    "NO ESTABA CASADO(A) Y LLEVABA MENOS DE DOS ANOS VIVIENDO CON SU PAREJA": "UNION LIBRE < 2 ANOS",
    "NO ESTA CASADA Y LLEVA MENOS DE DOS ANOS VIVIENDO CON SU PAREJA": "UNION LIBRE < 2 ANOS",
    "ESTABA SOLTERO(A)": "SOLTERA",
    "ESTA SOLTERA": "SOLTERA",
    "ESTABA CASADO(A)": "CASADA",
    "ESTA CASADA": "CASADA",
    "ESTABA SEPARADO(A), DIVORCIADO(A)": "SEPARADA O DIVORCIADA",
    "ESTA SEPARADA, DIVORCIADA": "SEPARADA O DIVORCIADA",
    "ESTABA VIUDO(A)": "VIUDA",
    "ESTA VIUDA": "VIUDA",
}

df_prep["estado_conyugal_madre"] = df_prep["estado_conyugal_madre"].replace(category_map)

In [ ]:
# 🔎 Check — six states plus the missing values, and the totals still add up
df_prep["estado_conyugal_madre"].value_counts(dropna=False)

`replace()` leaves anything not in the dictionary untouched, which is exactly what you want — a label you forgot survives in the output instead of vanishing. Compare the before and after counts: six states plus the missing values, and the totals still add up.

> Justification:

* Which labels did you merge, and what evidence says they mean the same thing?
* Did any merge destroy a distinction that might matter to your question?
* Is every original label accounted for in your dictionary?

...

## 4.4 Rare Categories

A category with 1 or 5 rows is not automatically an error. Before touching it, ask what it *is*:

- A **real but rare** state (`TRIPLE` births, `PALENQUERO DE SAN BASILIO` ethnicity) — usually keep. Rarity is information; a question about rare events depends entirely on these rows.
- A **fragment** of a larger category that the consolidation step missed — merge it.
- A **data entry error** (a value that cannot exist) — treat it in the plausibility step, not here.

Grouping rare levels into `OTRO` is a legitimate move, but it is a modeling convenience, and it *destroys* exactly the rows a rare-event question needs. Decide by what you asked, not by what looks tidy.

In [ ]:
# 🔧 Recipe — how thin is the tail? Swap in any categorical column.
variable = "pertenencia_etnica"

counts = df_prep[variable].value_counts(dropna=False)
counts[counts < 0.01 * len(df_prep)]

In [ ]:
# 🔧 Recipe — group rare levels into one "OTRO" category.
# Adopt this only when your question does not depend on the rare ones.
rare = counts[counts < 0.01 * len(df_prep)].index
print("would merge:", list(rare))

# df_prep[variable] = df_prep[variable].where(~df_prep[variable].isin(rare), "OTRO")

> Observations:

* Which rare categories are real, which are fragments, which are errors?
* Does your question depend on any of them?

...

## 4.5 Missing Values in Nominal Columns

Now that the disguised codes are unmasked (Section 3), decide what the remaining `NaN`s become. Three options, in rough order of how often they're right:

1. **Keep as an explicit category** (`"SIN DATO"`). Honest, keeps the rows, and makes "we don't know" visible in every chart. Best default for nominal data.
2. **Leave as `NaN`.** Fine when the column is peripheral to your question — but be aware that `groupby` drops these rows silently by default.
3. **Impute with the mode.** Rarely defensible: it invents membership in the *largest* group, which biases exactly the comparison you're most likely to make.

Never impute a nominal column just to make `isna().sum()` read zero.

In [ ]:
# 🔧 Recipe — make missingness an explicit, visible category.
# Shown without assigning back, so you can see the effect before adopting it.
localidad_labelled = df_prep["localidad"].fillna("SIN DATO")
localidad_labelled.value_counts().head(3)

# To adopt it, assign the result back to the column:
# df_prep["localidad"] = df_prep["localidad"].fillna("SIN DATO")

> 💡 Example (birth data): `localidad` is now ~29.6% missing. Filling it with the most common neighbourhood would fabricate a geographic distribution; labelling it `SIN DATO` keeps the rows and makes the gap impossible to overlook in Sessions 6–8. If a question depends on precise geography, the honest answer may be that this column can't support it.

> Justification:

* For each nominal column with missing values: which option, and why?
* Would your choice change if the column were central to your question rather than peripheral?

...

# 5. Ordinal Variables

**When you need this**: at least one of your columns has categories with a real order. If none do, skip to Section 6.

Categories with a meaningful order: education level, severity scales, satisfaction ratings, Apgar scores. Pandas has no way to know that order — you have to state it, and stating it is a **domain decision, not a data fact**.

## 5.1 Declaring the Order

Write the ladder out explicitly, from lowest to highest. Where two levels are genuinely parallel, or the ordering is arguable, say so — the ambiguity is real and hiding it doesn't make it go away.

The pattern never changes: `pd.Categorical(column, categories=[...], ordered=True)`. Swap in your column and your ladder.

In [ ]:
# 💡 Example (birth data): the mother's education level
df_prep["nivel_educativo_madre"].value_counts(dropna=False)

In [ ]:
# 🧱 Script
education_levels = [
    "NINGUNO",
    "PREESCOLAR",
    "BASICA PRIMARIA",
    "BASICA SECUNDARIA",
    "MEDIA ACADEMICA O CLASICA",
    "MEDIA TECNICA",
    "NORMALISTA",
    "TECNICA PROFESIONAL",
    "TECNOLOGICA",
    "PROFESIONAL",
    "ESPECIALIZACION",
    "MAESTRIA",
]

for col in ["nivel_educativo_madre", "nivel_educativo_padre"]:
    df_prep[col] = pd.Categorical(df_prep[col], categories=education_levels, ordered=True)

> 💡 Example (birth data): two of those placements are judgment calls. `MEDIA TECNICA` and `MEDIA ACADEMICA O CLASICA` are *parallel* tracks at the same level, not a rung apart — the ladder forces an order that reality doesn't have. `NORMALISTA` (teacher training) sits somewhere between secondary and tertiary depending on the era. Both were placed deliberately; a reader who disagrees can see exactly what was assumed and change one list.

> Justification:

* Where did you have to force an order that isn't really there?
* Would a different plausible ordering change any conclusion you plan to draw?

...

## 5.2 What the Order Buys You

In [ ]:
# 💡 Example (birth data): sorting and comparison now follow the ladder,
# not the alphabet
df_prep["nivel_educativo_madre"].value_counts().sort_index()

In [ ]:
# 💡 Example (birth data): comparisons work on ordered categoricals
(df_prep["nivel_educativo_madre"] >= "TECNICA PROFESIONAL").sum()

Two warnings before you go further:

- **Do not convert ordinal categories to integer codes here.** Codes (`0, 1, 2, …`) silently claim the gaps between levels are equal — that primary → secondary is the same "distance" as professional → master's. When a model needs numbers, that encoding decision belongs to the modeling step, with its own justification. The follow-on ML course covers it.
- **CSV does not preserve this.** Export to CSV and the ordered dtype is gone; re-reading gives you plain text again. See Section 10 for what to do about it.

...

## 5.3 Ordinal Variables That Look Numeric

Some ordinal variables arrive as numbers and are easy to mistreat. `apgar1`/`apgar2` (1–10 clinical scores) and `periodo_de_reporte` (quarters 1–4) are numbers you can average — but the average of a quarter number is meaningless, and the average of an Apgar score is only weakly meaningful.

Leave them numeric if you'll use them for ranking or thresholds; convert to ordered categoricals if you'll use them for grouping. Either way, note the decision — `describe()` will happily report a mean for both.

In [ ]:
# 🔧 Recipe — is this numeric column really a scale, a count, or a label?
# Swap in any numeric-looking column; a short, fixed set of values is the giveaway.
variable = "apgar1"

df_prep[variable].value_counts().sort_index()

In [ ]:
# 🔧 Recipe — if it is really ordinal, declare it the same way 5.1 did.
# Leave it numeric instead when you need thresholds or ranking.
apgar_levels = list(range(1, 11))

# df_prep[variable] = pd.Categorical(df_prep[variable], categories=apgar_levels, ordered=True)

> Observations:

* Which numeric columns in your dataset are actually ordinal or nominal labels?
* For each: does any statistic you plan to compute on it make sense?

...

# 6. Numeric Variables

**When you need this**: your diagnosis flagged impossible values, extreme values, or missing numbers.

Continuous measurements (`peso_gramos`, `talla_centimetros`) and discrete counts (`numero_embarazos`, `numero_consultas_prenatales`) share the same preparation order:

**repair what is impossible → decide about what is merely extreme → fill what is missing**

Doing it in any other order contaminates the statistics you use to make the next decision.

## 6.1 Plausibility Repair

Session 4 flagged values outside a plausible range. Values that *cannot be real* are not outliers — they are errors, and they should become `NaN` before any statistic is computed from the column.

**Repair the cell, not the row.** A record with one impossible value still has 30 perfectly good ones; deleting it throws away 30 valid observations to remove 1 bad one.

In [ ]:
# 🔧 Recipe — swap in a numeric column and the range you consider credible.
variable = "edad_madre"
plausible_min, plausible_max = 10, 60

implausible = ~df_prep[variable].between(plausible_min, plausible_max) & df_prep[variable].notna()
implausible.sum()

In [ ]:
# 💡 Example (birth data): fathers' ages
df_prep["edad_padre"].sort_values(ascending=False).head(8)

In [ ]:
# 🧱 Script
implausible = df_prep["edad_padre"].gt(70) & df_prep["edad_padre"].notna()

df_prep.loc[implausible, "edad_padre"] = np.nan

In [ ]:
# 🔎 Check — how many cells were repaired?
implausible.sum()

> 💡 Example (birth data): the two ages removed are 94 and 100. A 68-year-old father is unusual but possible, so the cutoff was set at 70 — a **judgment**, not a fact, which is exactly why it is written down as one number a reader can move and re-run.

> Justification:

* What range did you consider credible, and what is that based on — domain knowledge, documentation, or a guess?
* How many cells did you repair? If it's a large share, the column may have a systematic problem worth investigating instead.

...

## 6.2 Outliers Are Not Errors

An outlier is a *statistical* statement: far from the rest. An error is a *factual* statement: cannot be true. Treating the first as if it were the second is the most common way students damage a dataset in this course.

Before capping or removing anything, look for **corroborating evidence in another column**. Real extremes usually leave a trace elsewhere; data-entry errors usually stand alone.

In [ ]:
# 💡 Example (birth data): 48 babies weigh under 1000 g. Error, or real?
df_prep["peso_gramos"].describe()

In [ ]:
# 💡 Example (birth data): if these are real, gestation should be short too
pd.crosstab(df_prep["peso_gramos"] < 1000, df_prep["tiempo_de_gestacion"] < 32)

In [ ]:
# 💡 Example (birth data): and the two variables should move together overall
df_prep["peso_gramos"].corr(df_prep["tiempo_de_gestacion"]).round(3)

> 💡 Example (birth data): 47 of the 48 very-low-weight babies were also born before 32 weeks, and the two variables correlate at 0.77. These are real preterm births, not typos — and in a hospital dataset they are very likely the *most clinically important* rows in the file. Capping `peso_gramos` at the 1st percentile would have erased precisely the population a neonatal question is about.

The decision here is **leave** — and it belongs in your preparation log exactly like a repair would. "Investigated, decided to keep, here's the evidence" is a result. A log that only lists changes hides the hardest thinking you did.

**And it belongs in section 2b of your record as well.** This finding did not come from your diagnosis report, and it could not have: Session 4 checked whether birth weights were *possible*, not how they were *distributed*. Recording it as an emergent finding — with "could it have been diagnosed in Session 4? no, it needed the cleaned data" — is the honest account of how it was found.

When extremes *are* errors, or when they are real but distort a statistic you depend on, these are the two standard treatments. Both are here as recipes — use them when you can justify them, not by default.

In [ ]:
# 🔧 Recipe — capping at percentiles (winsorizing): keeps the row, limits the value.
variable = "peso_gramos"

lower = df_prep[variable].quantile(0.01)
upper = df_prep[variable].quantile(0.99)
print(f"would clip to [{lower}, {upper}]")

# df_prep[variable] = df_prep[variable].clip(lower, upper)

In [ ]:
# 🔧 Recipe — the IQR rule: a detection tool, not an instruction to delete.
variable = "peso_gramos"

q1 = df_prep[variable].quantile(0.25)
q3 = df_prep[variable].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print(f"IQR fence: [{lower_bound}, {upper_bound}]")

outliers = ~df_prep[variable].between(lower_bound, upper_bound)
outliers.sum()

> Justification:

* For each extreme value you found: error, or real?
* What evidence in *another* column supports that call?
* If you capped or removed anything, what does that do to the question you're trying to answer?

...

## 6.3 A Harder Case: Real, Uncomfortable Values

Some values are implausible-looking, fully real, and central to why the data was collected at all.

> 💡 Example (birth data): 47 mothers are under 15 years old, the youngest 12. Nothing is wrong with those records. Adolescent pregnancy is a documented public-health reality, and these rows are exactly the ones a public-health question would be about. Removing them as "outliers" would delete the finding, not clean the data.

The test is never "does this value look extreme?" It is **"is there a reason to believe this measurement is wrong?"** Discomfort is not evidence.

> Your turn: does your dataset contain values that are extreme *and* real? What would be lost by removing them?

...

## 6.4 Missing Numeric Values

Now — and only now, with impossible values already turned into `NaN` — fill what's missing. Use the diagnosis from Section 3.2: structural, random, or informative. Each gets a different treatment.

### Structural missingness: fill with what the structure implies

In [ ]:
# 🧱 Script — 💡 birth data: the value is missing exactly where the mother has
# no education, because "years approved" of nothing is zero, not unknown.
structural = (
    df_prep["nivel_educativo_madre"].eq("NINGUNO")
    & df_prep["ultimo_ano_aprobado_madre"].isna()
)

df_prep.loc[structural, "ultimo_ano_aprobado_madre"] = 0

In [ ]:
# 🔎 Check — how many rows did that fill?
structural.sum()

> 💡 Example (birth data): the remaining missing values in this column sit where `nivel_educativo_madre` is itself unknown. There, `0` would be a fabrication — the education level might be anything. Those stay `NaN`. One column, two causes of missingness, two different correct answers.

### Random missingness: impute, or drop if trivial

In [ ]:
# 💡 Example (birth data): what's still missing, and how much?
(100 * df_prep.isna().mean()).sort_values(ascending=False).head(8).round(2)

In [ ]:
# 🧱 Script — a handful of rows with missing clinical measurements
df_prep = df_prep.dropna(subset=["peso_gramos", "talla_centimetros", "tiempo_de_gestacion", "apgar1", "apgar2"])

In [ ]:
# 🔎 Check — rows lost so far, measured against the raw file
print(f"{len(df_raw)} raw rows -> {len(df_prep)} now")

Dropping is defensible here for one reason only: the share is negligible *and* the missing values are scattered rather than concentrated in one kind of record. Check both before you use this pattern — if the dropped rows all came from one hospital, one month, or one category, you have just introduced a bias while "cleaning."

In [ ]:
# 🔧 Recipe — median imputation, when dropping would cost too much.
variable = "talla_centimetros"
print("would fill with:", df_prep[variable].median())

# Median is preferred over mean: it is not dragged by the extremes you kept.
# df_prep[variable] = df_prep[variable].fillna(df_prep[variable].median())

In [ ]:
# 🔧 Recipe — group-aware imputation: usually more defensible than a global median.
variable = "ultimo_ano_aprobado_madre"
group_col = "nivel_educativo_madre"

df_prep.groupby(group_col, observed=True)[variable].median()

# df_prep[variable] = df_prep[variable].fillna(
#     df_prep.groupby(group_col, observed=True)[variable].transform("median")
# )

### Informative missingness: keep the gap, add a flag

In [ ]:
# 🧱 Script
df_prep["edad_padre_faltante"] = df_prep["edad_padre"].isna()

In [ ]:
# 🔎 Check — what share of records is flagged?
100 * df_prep["edad_padre_faltante"].mean()

> 💡 Example (birth data): the father's age is missing in about 4.5% of records, and the father's education level in about 19%. Both are *about the father* — a pattern suggesting the form was often completed without him present. That pattern is data. Filling in a median age would delete it and replace it with a fiction.

> Justification:

* For each numeric column you filled: which cause, which method, and what does the fill assume?
* Which columns did you deliberately leave missing, and why is that the better answer?

...

## 6.5 Derived Variables (a first pass)

Preparation can also *add* columns — binning a continuous variable into meaningful groups, combining two columns into a rate, extracting a component from a date. This is enrichment, and it is as much a part of preparation as cleaning.

Keep this pass light. Most enrichment ideas come *from* exploration, and Sessions 6–8 will send you back here with better ones than you can invent today.

In [ ]:
# 🔧 Recipe — bin a continuous variable using domain-meaningful cut points.
# One fewer label than boundaries, always.
variable = "numero_consultas_prenatales"
bins = [-1, 0, 3, 7, np.inf]
labels = ["ninguna", "1-3", "4-7", "8+"]

pd.cut(df_prep[variable], bins=bins, labels=labels).value_counts().sort_index()

# df_prep[f"{variable}_grupo"] = pd.cut(df_prep[variable], bins=bins, labels=labels)

In [ ]:
# 🧱 Script — 💡 Example (birth data): maternal age groups used in public-health reporting
df_prep["edad_madre_grupo"] = pd.cut(
    df_prep["edad_madre"],
    bins=[0, 14, 19, 34, np.inf],
    labels=["<15", "15-19", "20-34", "35+"],
)

In [ ]:
# 🔎 Check
df_prep["edad_madre_grupo"].value_counts().sort_index()

Note that the original column is **kept**. Binning throws away information, so it should add a column, never replace one — if the bands turn out to be wrong in Session 7, you want the raw values still there.

> Justification:

* Where did your cut points come from? (Equal-width bins chosen by convenience are rarely meaningful.)
* What does the derived column let you ask that the raw one didn't?

...

# 7. Temporal Variables

**When you need this**: your dataset has a date or time column. If it doesn't, skip to Section 8.

Dates arrive as text and stay useless until parsed. Once parsed, they give you components (year, month, weekday), ordering, and duration arithmetic — none of which work on a string.

## 7.1 Parsing

In [ ]:
# 🔧 Recipe — swap in your own date column. `format` describes how the text is
# written; "ISO8601" covers the 2022-01-01T00:00:00-05:00 style used here, and
# "mixed" lets pandas infer when your file is less consistent.
variable = "fecha_nacimiento"

# df_prep[variable] = pd.to_datetime(df_prep[variable], format="ISO8601", errors="coerce")

In [ ]:
# 💡 Example (birth data): ISO 8601 with a UTC-05:00 offset
df_prep["fecha_nacimiento"].head(3)

In [ ]:
# 🧱 Script
df_prep["fecha_nacimiento"] = pd.to_datetime(
    df_prep["fecha_nacimiento"], format="ISO8601", errors="coerce"
)

In [ ]:
# 🔎 Check — the dtype should now be datetime, not str
df_prep["fecha_nacimiento"].dtype

`errors="coerce"` turns unparseable values into `NaT` instead of raising. That is the right choice **only if you then check how many appeared** — otherwise it silently converts a format problem into missing data.

In [ ]:
# 🔧 Recipe: how many values failed to parse?
df_prep["fecha_nacimiento"].isna().sum()

...

## 7.2 Coverage and Redundancy

In [ ]:
# 💡 Example (birth data): what period does this extract actually cover?
df_prep["fecha_nacimiento"].agg(["min", "max"])

In [ ]:
# 💡 Example (birth data): does the standalone year column agree with the date?
(df_prep["fecha_nacimiento"].dt.year != df_prep["ano"]).sum()

> 💡 Example (birth data): the extract runs from 2022-01-01 to 2023-03-31 — five quarters, not two full years. Any per-year comparison would be comparing 12 months against 3. That is a **scope fact** for your EDA, not a defect to fix.

> And `ano` never disagrees with the parsed date, which makes it a derived column rather than an independent one. Keeping it is harmless; knowing it is redundant matters, because agreement between the two proves nothing.

In [ ]:
# 🔧 Recipe: extract the components you'll actually group by
# df_prep["anio"] = df_prep["fecha_nacimiento"].dt.year
# df_prep["mes"] = df_prep["fecha_nacimiento"].dt.month
# df_prep["dia_semana"] = df_prep["fecha_nacimiento"].dt.day_name()

> Observations:

* What period does your dataset cover, and is it complete at both ends?
* Do any two of your columns encode the same time information?

...

# 8. Cross-Field Consistency

**When you need this**: two columns in your dataset are related by a rule that should always hold — and your diagnosis suspected it doesn't.

Every check so far looked at one column at a time. Session 4 raised this as a prompt without templating it, because the rules are dataset-specific — but the pattern is always the same:

**pick two related columns → write down the rule that must hold → count the rows that break it → decide.**

The decision is the interesting part. When a rule is violated you usually know that *something* is wrong, but not *which* of the two columns is the wrong one. That is precisely when **flag** beats **repair**.

In [ ]:
# 🔧 Recipe: state the rule as code, then count the violations
# rule_violated = df_prep["column_a"] > df_prep["column_b"]
# rule_violated.sum()

In [ ]:
# 🧱 Script — 💡 birth data: a mother cannot have more live births than pregnancies
rule_violated = df_prep["numero_hijos_nacidos_vivos"] > df_prep["numero_embarazos"]

df_prep["paridad_inconsistente"] = rule_violated

In [ ]:
# 🔎 Check — how many rows break the rule, and what do they look like?
print(f"{rule_violated.sum()} rows ({100 * rule_violated.mean():.1f}%)")
df_prep.loc[rule_violated, ["numero_hijos_nacidos_vivos", "numero_embarazos"]].head()

> 💡 Example (birth data): 116 rows (1.9%) report more live births than pregnancies. Both fields are self-reported at admission, and either could be the mistaken one. Repairing would mean guessing; dropping would remove 116 otherwise-complete records. The flag keeps the rows usable and lets any later analysis exclude them *explicitly* — `df[~df["paridad_inconsistente"]]` — which is a visible choice rather than a silent one.

> Justification:

* Which rules should hold between columns in your dataset? (Dates in order, totals equal to their parts, counts within capacity, geography consistent.)
* For each violation: can you tell which column is wrong? If not, flag it.

...

# 9. Export & Handoff to the Script

## 9.1 Export the Prepared Dataset

Before you write anything, look at what you are about to hand over. You changed this dataset a lot; the last thing you should do is confirm it still looks like the thing you meant to build.

In [ ]:
# 🔎 Check
df_prep.info()

In [ ]:
# 🔎 Check — every gap left here should be one you left on purpose
(100 * df_prep.isna().mean()).sort_values(ascending=False).head(10).round(2)

In [ ]:
# 🔎 Check
df_prep.head()

> Ask yourself, before exporting:
>
> * Are the dtypes what you intended — dates as datetime, ordinals as ordered categoricals?
> * Is every remaining missing value a decision you can name, and not something you overlooked?
> * Did the row count change only as much as you decided it should?
> * Would the shape of this table surprise the person who wrote your diagnosis report?

Prepared data goes to `data/processed/`, never back into `data/raw/`. The raw file stays exactly as acquired, so that the whole pipeline can always be re-run from the original.

In [ ]:
# 🧱 Script
PROCESSED_PATH = "../data/processed/nacimientos_preparados.csv"

df_prep.to_csv(PROCESSED_PATH, index=False)
print(f"wrote {len(df_prep)} rows to {PROCESSED_PATH}")

**A caveat about CSV.** It stores text, so it cannot preserve the work you did on dtypes: ordered categoricals become plain strings again, datetimes become text, booleans become `True`/`False` strings. Re-reading this file in Session 6 gives you a DataFrame that *looks* right and has lost its type information.

Two ways to live with that:

1. Export to Parquet as well (`df_prep.to_parquet(...)`), which preserves dtypes exactly — this is why practitioners prefer it for intermediate data.
2. Keep CSV for portability and let your preparation **script** be the thing that restores the dtypes, since it re-applies them every time it runs.

Option 2 is the one this course takes, and it is the strongest argument for why the script exists at all.

In [ ]:
# 🔧 Recipe: dtype-preserving export (optional)
# df_prep.to_parquet("../data/processed/nacimientos_preparados.parquet", index=False)

...

## 9.2 Closing the Record

By now sections 1, 2 and 3 of your preparation record should already be written — the plan before you started, the revisions and the log as you went. This is where you collect the log properly and close out the last two sections.

**Section 3 — the log.** Diagnosis said what was wrong; the plan said what you intended; the log says what you **actually did**, and why, including the steps where the action was "nothing, deliberately.

💡 **Example** — for the birth-records dataset:

| Column | What I did | Rows affected | Why |
|---|---|---|---|
| `pais_residencia`, `profesion_certificador` | dropped (constant) | 0 | one value for every record; kept as scope facts instead |
| all rows | checked for duplicates, none found; `drop_duplicates()` kept as a guard | 0 | no exact or key repeats; the line protects against a re-appended extract |
| all text columns | recoded `SIN INFORMACION` and similar codes to `NaN` | 4,094 cells | these mean "unknown"; counting them as categories hides true missingness |
| all text columns | stripped whitespace, upper-cased | 6,278 | prevents one category being counted as several |
| `pertenencia_etnica` | merged two labels for the same group | 15 | the two differed only in punctuation |
| `estado_conyugal_madre` | consolidated 12 labels into 6 states | 6,218 | past/present wording is a form artifact |
| `nivel_educativo_madre`, `nivel_educativo_padre` | declared as ordered categorical | 6,278 | the order is meaningful and sorting should respect it |
| `edad_padre` | ages > 70 → `NaN`; missing indicator added; left unfilled | 2 | not credible; and whether the age was recorded may itself be informative |
| `peso_gramos` | **nothing** — kept the extreme low values | 0 | corroborated by gestation weeks (47/48 born < 32 weeks): real preterm births |
| `ultimo_ano_aprobado_madre` | filled with 0 where education is `NINGUNO` | 44 | structurally not applicable, not unknown — zero is the true value |
| clinical measurements | dropped rows with missing values | 14 | 0.2% of rows; imputing five clinical variables invents more than it recovers |
| `edad_madre_grupo` | derived from `edad_madre` | 6,264 | standard public-health bands; the raw age is kept as well |
| `numero_hijos_nacidos_vivos`, `numero_embarazos` | flagged inconsistent rows | 116 | impossible combination, but no way to tell which field is wrong |

The row for `peso_gramos` is the one to notice: the action was **nothing**, and it still earns a line. Investigating and deciding to keep is a result. A log that lists only changes hides the hardest thinking you did.

**Section 4 — plan vs. log.** Read the log against the plan and answer three questions:

* Did you do everything you planned?
* Did you do anything you *didn't* plan? (That's fine — but it belongs in the log, with a reason.)
* Did any decision change between the two? Say what changed your mind.

**Section 5 — result.** Rows and columns in and out, which missing values you left on purpose and why, and what you checked before exporting.

### Before you submit

The record is finished when all five of these are true:

- [ ] every finding in your diagnosis report appears in section 1 with a decision — repair, flag, or leave
- [ ] every 🧱 step in your script appears in section 3
- [ ] anything that revised the diagnosis is in section 2, and the diagnosis report itself is **unchanged**
- [ ] sections 4 and 5 are filled in
- [ ] a reader who has never seen your dataset could follow the record from diagnosis to prepared file without asking you a question

If nothing in section 3 says "nothing, deliberately", go back and check whether you ever really considered leaving something alone — or whether you cleaned by reflex.

...

## 9.3 Extracting the Script

This is the second half of Session 5's deliverable, and the point of every 🧱 marker above.

**What goes in**: the 🧱 cells, in the order they appear — load, structural fixes, unmasking, per-type preparation, cross-field flags, export. Each one is self-contained: every variable a 🧱 cell uses is defined in a 🧱 cell, so copying them in sequence gives you code that runs.

**What stays out**: everything you did to *decide* or to *look* — every 🔎 **Check** cell, every 🔧 **Recipe** you didn't adopt, plus `value_counts()`, `describe()`, crosstabs and dead ends. Those were the reasoning; the script is the conclusion.

That separation is why the 🧱/🔎 split exists in this notebook at all. In a notebook, printing the result of a step is how you know it worked; in a script, nobody is watching, and the output would only scroll past in a log. So the 🔎 cells stay behind — which puts the burden on you to have looked at them properly *before* the logic became a script.

A workable shape:

```python
"""Prepare the raw birth-records dataset for exploratory analysis."""

import pandas as pd

RAW_PATH = "data/raw/nacimientos_ocurridos_hospital_general_medellin.csv"
PROCESSED_PATH = "data/processed/nacimientos_preparados.csv"
MISSING_CODES = ["SIN INFORMACION", "NO REPORTA", ...]


def load_raw(path: str) -> pd.DataFrame:
    ...


def unmask_missing_codes(df: pd.DataFrame) -> pd.DataFrame:
    ...


def prepare_categorical(df: pd.DataFrame) -> pd.DataFrame:
    ...


def prepare_numeric(df: pd.DataFrame) -> pd.DataFrame:
    ...


def main() -> None:
    df_raw = load_raw(RAW_PATH)

    df_prep = df_raw.copy()
    df_prep = unmask_missing_codes(df_prep)
    df_prep = prepare_categorical(df_prep)
    df_prep = prepare_numeric(df_prep)

    df_prep.to_csv(PROCESSED_PATH, index=False)
    print(f"wrote {len(df_prep)} rows to {PROCESSED_PATH}")


if __name__ == "__main__":
    main()
```

Three rules for the extraction:

1. **Each function does one stage** and returns a DataFrame — that is what makes the pipeline readable and each stage testable.
2. **Every threshold is a named constant at the top**, not a number buried in the middle. Your reviewer should be able to change the age cutoff without reading the whole file.
3. **The script only ever writes to `data/processed/`** — it reads the raw file and never modifies it, so the whole pipeline can be re-run from the original at any time.

**A worked example of all of this** lives in `templates/4-preparation_script_template.py` — the birth-records pipeline above, already extracted into exactly this shape. Read it alongside your own extraction; it is the same logic you have been running, with the reasoning removed.

**How to prove the extraction worked**: run your script, re-read the file it wrote, and compare it against this notebook's `df_prep`. Same shape, same columns, same missingness. If they differ, something you were relying on lived in a 🔎 Check cell and never made it across.

In [ ]:
# 🔧 Recipe: verify the script reproduces the notebook
# df_from_script = pd.read_csv(PROCESSED_PATH)
# print(df_from_script.shape == df_prep.shape)
# print(sorted(df_from_script.columns) == sorted(df_prep.columns))

...

## 9.4 Next: Sessions 6–8 — Exploratory Data Analysis

You now have a prepared dataset, a log of what you did to it, and a script that reproduces it from the raw file.

None of that is final. EDA exists partly to *test* your preparation: a distribution with a suspicious spike at the median usually means an imputation you shouldn't have made; a category that dominates every chart is often two categories you merged too eagerly; a relationship that fails to appear may be hidden by a column you dropped. When exploration sends you back here, that is the CRISP-DM loop from Sessions 1–2 working exactly as drawn — not a mistake.

And it works in the other direction too: EDA will suggest **enrichments** — new derived columns, better groupings — that belong in the script alongside the fixes.

**Bring to Session 6**: this notebook, your preparation script, your preparation log, and the question you wrote in Session 4. You'll be asked at the end of Session 8 whether the data answered it — or changed it.

...

# Appendix — Removing Accents

Section 4.1 handled case and whitespace with `.str.strip().str.upper()`. Accents are the remaining variant of the same problem: `"INDÍGENA"` and `"INDIGENA"` are one category typed two ways, and upper-casing leaves them apart.

There is no one-line pandas fix, because "strip the accent, keep the letter" is a Unicode operation rather than a text one. The function below does it; you can use it without reading it closely — what matters is *why* you're calling it, not how it works inside.

Use it only if your own dataset has accented text. This course's birth-records extract does not.

In [ ]:
# 🔧 Recipe (optional): use only if your dataset has accented categories
import unicodedata


def remove_accents(text: str) -> str:
    """Return the text with accents removed, leaving the base letters unchanged.

    Args:
        text: A text value, e.g. "INDÍGENA".

    Returns:
        The same text without accent marks, e.g. "INDIGENA".
    """
    decomposed = unicodedata.normalize("NFKD", text)
    return "".join(char for char in decomposed if not unicodedata.combining(char))


# df_prep[variable] = df_prep[variable].map(remove_accents, na_action="ignore")

> A word of caution: this merges `"PEÑA"` into `"PENA"` — two different Spanish words. Accent removal is a *hygiene* step for matching categories, not a way to store names. Apply it to the columns you group by, not to every text column in your dataset.